# Feature Extraction


In [ ]:
from pathlib import Path
import os, sys, copy, tempfile
sys.dont_write_bytecode = True
import numpy as np
import nibabel as nib
import torch
root = Path.cwd()
if not (root / 'configs').is_dir(): root = root.parent
assert (root / 'configs/pretrain.yaml').is_file(), 'Run from the repository root or notebooks directory'
sys.path.insert(0, str(root))
torch.set_num_threads(2)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
dino_weight = Path(os.environ.get('DINO_CHECKPOINT', root / 'weights/dinov3/teacher_checkpoint.pth'))
sampler_weight = Path(os.environ.get('SAMPLER_CHECKPOINT', root / 'weights/sampler/best_all8_hit.pt'))
assert dino_weight.is_file() and sampler_weight.is_file(), 'Set DINO_CHECKPOINT and SAMPLER_CHECKPOINT to existing checkpoints'
from features.extractor import load_backbone, normalize_uint8, make_feature_crops, encode_crops
from lesion_prior.config import load_config
from lesion_prior.checkpoint import build_model_from_checkpoint
from lesion_prior.dataset import LesionPriorDataset
from lesion_prior.infer_sw import sliding_window_predict
from scripts.select_local_crops import select_boxes
from processed_data.build_unlabeled_proposal_points import convert_one_image
backbone = load_backbone(dino_weight, root / 'configs/pretrain.yaml', device=device)
sampler_cfg = load_config(str(root / 'configs/crop_light_train.yaml'))
sampler = build_model_from_checkpoint(sampler_weight, sampler_cfg).to(device).eval()
assert backbone.embed_dim == 1024
print({'device': device, 'DINO_embedding_dim': backbone.embed_dim, 'sampler_strict_load': True})


## Sampler


In [ ]:
tmp = tempfile.TemporaryDirectory()
image_path = Path(tmp.name) / 'synthetic.nii.gz'
rng = np.random.default_rng(42)
raw = rng.integers(1, 255, size=(64, 72, 80), dtype=np.uint8)
affine = np.array([[0, -1.5, 0, 120], [0, 0, 1.5, -20], [-1.5, 0, 0, 140], [0,0,0,1.]])
nib.save(nib.Nifti1Image(raw, affine), image_path)
cfg = copy.deepcopy(sampler_cfg)
cfg.data.sw_roi_size = (32, 32, 32)
cfg.data.cache_dir = ''
row = {'case_id':'synthetic_demo', 'image_path':str(image_path), 'seg_path':'', 'modalities':'CT', 'split':'demo'}
dataset = LesionPriorDataset(cfg.data, cfg.target, split='demo', manifest_rows=[row], augment=False, compute_targets=False)
sample = dataset[0]
prob = sliding_window_predict(sampler, sample['image'], sample['mod_mask'], cfg, torch.device(device), amp=device=='cuda', sw_batch_size=1)
assert torch.isfinite(prob).all()
print({'probability_grid': list(prob.shape), 'range': [float(prob.min()), float(prob.max())]})


In [ ]:
centers = []
counts = {}
for selector in ['gaussian', 'nms']:
    boxes = select_boxes(prob, sample['image'].shape[1:], cfg, selector)
    entry, stats = convert_one_image(str(image_path), ['CT'], [b.to_dict() for b in boxes],
        target_spacing=cfg.data.spacing, token_stride=cfg.target.token_stride, topk=8,
        case_id='synthetic_demo', selector=selector, source_crop_size=cfg.crop_eval.local_crop_size,
        checkpoint_name='best_all8_hit.pt')
    assert entry is not None and entry['raw_shape'] == list(raw.shape)
    centers.extend(entry['proposal_centers'])
    counts[selector] = len(entry['proposal_centers'])
centers = np.unique(np.asarray(centers), axis=0)
assert (centers >= 0).all() and (centers < np.asarray(raw.shape)).all()
print({'raw_voxel_centers': len(centers), 'selectors': counts})


## DINO Features


In [ ]:
normalized = normalize_uint8(raw)
crops = make_feature_crops(normalized, centers, seed=42, case_id='synthetic_demo')
assert crops.shape == (10, 1, 128, 128, 128)
features = encode_crops(backbone, crops, batch_size=1)['cls']
assert features.shape == (10, 1024) and torch.isfinite(features).all()
patch_tokens = encode_crops(backbone, crops[:1], return_patch_tokens=True)['patch_tokens']
assert patch_tokens.shape == (1, 512, 1024)
print({'views':list(features.shape), 'patch_tokens':list(patch_tokens.shape), 'all_finite':True})


In [ ]:
print({'feature_shape': list(features.shape), 'downstream_ready': True})
tmp.cleanup()


## Downstream

[Commands](../README.md#fine-tuning)
